# Submission: exp16 ResNet-18 downsampling, all thresholds 0.5

Card: [exp16_downsampling.md](../../experiments/exp16_downsampling.md)


In [ ]:
import os
import tempfile
from pathlib import Path

import pandas as pd

from core.config import ROOT
from core.cv import Config, CrossValidator
from core.data import read_test
from core.submission import Single

THRESHOLDS = (0.5, 0.5, 0.5, 0.5, 0.5)  # per class, CLASSES order
SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-downsampling-smoke",
        "folds": [0],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    TEST = read_test().head(64)
    PATH = Path(tempfile.mkdtemp()) / "smoke.csv"
else:
    OVERRIDES = {}
    TEST = None
    PATH = None
CONFIG = Config.from_toml(
    ROOT / "sections" / "model" / "configs" / "exp16_downsampling" / "resnet18.toml",
    **OVERRIDES
)

## Training


In [ ]:
CrossValidator(CONFIG).run()  # finished folds restored from checkpoints

## Predict


In [ ]:
submission = Single(CONFIG, path=PATH, fixed_thresholds=THRESHOLDS)
path = submission.write(TEST)
pd.read_csv(path).iloc[:, 1:].mean().round(3)  # predicted positive rate per class

## Upload


In [ ]:
if not SMOKE:
    submission.submit(
        "exp16 ResNet-18 downsample 0.2, folds 0-1, all thresholds 0.5",
    )